# 第 12 章习题与解答

> 本章习题聚焦 DPO 的**数学直觉**和**工程细节**。建议先自己想,再展开答案。

## Exercise 12.1(易)

**题目**:DPO 损失中的 $\beta$ 参数起什么作用?如果 $\beta = 1.0$ 和 $\beta = 0.01$,训练行为会有什么区别?

<details><summary><b>参考答案</b></summary>

$\beta$ 是 **KL 约束温度**,控制策略允许偏离参考模型的程度。

DPO loss 为 $-\log\sigma(\beta \cdot \text{logits})$,其中 logits 是策略相对于 ref 的偏好提升量。

| $\beta$ | 约束强度 | 训练行为 | 风险 |
|---|---|---|---|
| **大**(如 1.0) | 弱 | $\beta \cdot \text{logits}$ 放大,小提升就能让 sigmoid 饱和,loss 快速下降。模型倾向于大幅偏离 ref | 过拟合、灾难性遗忘、mode collapse |
| **小**(如 0.01) | 强 | $\beta \cdot \text{logits}$ 缩小,需要很大的 logits 才能让 loss 下降。模型几乎不动 | 学不到偏好,训练无效 |
| **适中**(0.15) | 平衡 | 既能学到偏好,又不会偏离太远 | — |

数学上,$\beta$ 来源于 RLHF 目标中的 KL 项 $\beta \cdot \text{KL}(\pi \| \pi_{\text{ref}})$。$\beta$ 越大,KL 约束越弱(允许偏离),$\beta$ 越小约束越强(必须贴近 ref)。

> minimind 选 0.15 是经验值。原 DPO 论文用 0.1,大多数实现用 0.1~0.5。

</details>

In [ ]:
# 可视化 beta 对 loss 曲线的影响
import torch
import torch.nn.functional as F
import numpy as np

logits_range = np.linspace(-3, 5, 100)
print(f"{'logits':>8} | ", end="")
for beta in [0.01, 0.05, 0.15, 0.5, 1.0]:
    print(f"β={beta:<5}", end=" | ")
print()
print("-" * 70)

for logits_val in [-2, -1, 0, 1, 2, 3]:
    print(f"{logits_val:>8.1f} | ", end="")
    for beta in [0.01, 0.05, 0.15, 0.5, 1.0]:
        lp = -F.logsigmoid(torch.tensor(beta * logits_val)).item()
        print(f"{lp:>7.4f}", end=" | ")
    print()

print("\n观察:β=1.0 时 logits=1 的 loss 已经很低(0.31),")
print("      β=0.01 时 logits=1 的 loss 仍接近最大值(0.69),")
print("      说明 β 小时需要更大的 logits 提升才能降低 loss。")

## Exercise 12.2(中)

**题目**:为什么 DPO 不需要训练一个独立的 reward model?DPO 中的「隐式 reward」是什么?

<details><summary><b>参考答案</b></summary>

**RLHF 的流程**:

```
偏好数据 → 训练 reward model r(x,y) → PPO 用 r 优化策略 π
```

这里 reward model 是一个**独立的神经网络**,需要单独训练。它的输入是 $(x, y)$,输出是一个标量 reward。

**DPO 的推导**告诉我们:RLHF 的最优策略有闭式解:

$$r(x, y) = \beta \log \frac{\pi(y \mid x)}{\pi_{\text{ref}}(y \mid x)} + \beta \log Z(x)$$

也就是说,**reward 可以直接从策略模型的 log-prob 中读出来**!不需要训练一个独立的 reward model。

这就是「**隐式 reward**」:

$$\hat{r}(x, y) = \beta \log \frac{\pi(y \mid x)}{\pi_{\text{ref}}(y \mid x)}$$

DPO 训练时,策略模型 $\pi$ 同时扮演了两个角色:
1. **策略本身**:生成文本
2. **隐式 reward model**:通过 log-ratio 提供 reward 信号

当 $\pi(y \mid x) > \pi_{\text{ref}}(y \mid x)$ 时,隐式 reward 为正(策略认为这个回答比 SFT 更好);反之为负。

> **代价**:虽然不需要训练 reward model,但 DPO 需要在显存中同时保留 policy 和 ref model(2 倍显存)。另外,隐式 reward 的质量完全依赖策略模型本身的表达能力,不如独立 reward model 灵活。

</details>

In [ ]:
# 演示隐式 reward 的计算
import torch

# 模拟 ref 和 policy 对 chosen/rejected 的 log-prob
beta = 0.15

ref_chosen_lp   = torch.tensor([-5.0, -5.0])
ref_rejected_lp = torch.tensor([-6.0, -6.0])
pol_chosen_lp   = torch.tensor([-4.5, -4.8])   # policy 比 ref 更高估 chosen
pol_rejected_lp = torch.tensor([-6.5, -7.0])   # policy 比 ref 更低估 rejected

# 隐式 reward = beta * log(pi / pi_ref) = beta * (log_pi - log_pi_ref)
implicit_reward_chosen   = beta * (pol_chosen_lp - ref_chosen_lp)
implicit_reward_rejected = beta * (pol_rejected_lp - ref_rejected_lp)

print("隐式 reward:")
print(f"  chosen:   {implicit_reward_chosen.tolist()}   (正 = 策略偏好高于 ref)")
print(f"  rejected: {implicit_reward_rejected.tolist()} (负 = 策略偏好低于 ref)")
print(f"  reward 差 (chosen - rejected): {(implicit_reward_chosen - implicit_reward_rejected).tolist()}")
print(f"\n这正是 DPO logits * beta 的来源:")
print(f"  β * [(logπ(yw) - logπ(yl)) - (logπ_ref(yw) - logπ_ref(yl))]")
print(f"  = β * (logπ(yw) - logπ_ref(yw)) - β * (logπ(yl) - logπ_ref(yl))")
print(f"  = r̂(yw) - r̂(yl)  ← 就是隐式 reward 的差!")

## Exercise 12.3(难)

**题目**:如果偏好数据中 chosen 和 rejected 的质量差距很小(比如两个回答都还不错,只是 chosen 稍好一点),DPO 还能有效学习吗?从梯度信号的角度分析。

<details><summary><b>参考答案</b></summary>

**结论**:能学,但效率低。质量差距越小,梯度信号越弱,训练越慢。

**分析**:

DPO loss 为 $\mathcal{L} = -\log\sigma(\beta \cdot \text{logits})$,其中 logits 是策略相对于 ref 的偏好提升量。

梯度(对策略参数 $\theta$):

$$\frac{\partial \mathcal{L}}{\partial \theta} = -\sigma(-\beta \cdot \text{logits}) \cdot \beta \cdot \frac{\partial \text{logits}}{\partial \theta}$$

其中 $\sigma(-\beta \cdot \text{logits})$ 是梯度缩放因子。关键在于 **logits 的大小**:

1. **质量差距大**(chosen 明显好于 rejected):
   - 训练前 ref 对 chosen 的 log-prob 就明显高于 rejected(模型本身也更「喜欢」chosen)
   - $\log\pi_{\text{ref}}(y_w) - \log\pi_{\text{ref}}(y_l)$ 已经为正
   - 但初始 logits = 0(policy == ref),所以 $\sigma(0) = 0.5$,梯度正常

2. **质量差距小**(chosen 和 rejected 差不多):
   - $\log\pi_{\text{ref}}(y_w) \approx \log\pi_{\text{ref}}(y_l)$
   - ref 模型本身就**分不清**哪个更好
   - 两者概率接近,策略需要学到非常细微的差异才能让 logits > 0
   - $\frac{\partial \text{logits}}{\partial \theta}$ 很小(因为 chosen 和 rejected 的表示太接近)
   - **有效梯度极小**,训练收敛慢

3. **极端情况**(chosen 和 rejected 完全一样):
   - logits 恒为 0,$\frac{\partial \text{logits}}{\partial \theta} = 0$
   - **梯度为零**,完全学不到东西

**实践建议**:
- DPO 数据的 chosen/rejected 应该有**明确的质量梯度**,避免模糊的偏好标注
- 如果数据质量差距小,可以增大 $\beta$(放宽约束),或增加训练步数
- 原始 DPO 论文也讨论了这个问题,后续工作(IPO、KTO 等)试图改进小差距场景的稳定性

> 这也是为什么 DPO 的数据质量极其重要 —— 模糊的偏好标注会导致训练效率低下,甚至引入噪声。

</details>

In [ ]:
# 演示质量差距对梯度信号的影响
import torch
import torch.nn.functional as F

def dpo_grad_scale(chosen_ref_lp, rejected_ref_lp, beta=0.15):
    """
    计算梯度缩放因子 sigma(-beta * logits)
    训练初始:policy == ref,所以 logits = 0
    但 ref 本身对 chosen/rejected 的偏好差会影响后续训练
    """
    # 训练初始 logits = 0,所以 grad_scale = sigma(0) = 0.5
    # 但关键是 ref_margin = logp_ref(chosen) - logp_ref(rejected)
    # 这个值反映了「ref 模型本身是否区分得开」
    ref_margin = (chosen_ref_lp - rejected_ref_lp).item()
    return ref_margin

print("=== ref 模型对 chosen/rejected 的概率差 vs 学习难度 ===\n")
print(f"{'场景':>16} | {'ref_margin':>10} | {'学习难度':>10} | {'说明'}")
print("-" * 75)

# 场景1:差距大
print(f"{'差距大':>16} | {dpo_grad_scale(torch.tensor([-5.0]), torch.tensor([-15.0])):>+10.1f} | {'低':>10} | ref 能明确区分,梯度清晰")

# 场景2:差距中等
print(f"{'差距中等':>16} | {dpo_grad_scale(torch.tensor([-5.0]), torch.tensor([-7.0])):>+10.1f} | {'中':>10} | ref 能区分但不太确定")

# 场景3:差距小
print(f"{'差距小':>16} | {dpo_grad_scale(torch.tensor([-5.0]), torch.tensor([-5.3])):>+10.1f} | {'高':>10} | ref 几乎分不清,梯度弱")

# 场景4:无差距
print(f"{'完全一样':>16} | {dpo_grad_scale(torch.tensor([-5.0]), torch.tensor([-5.0])):>+10.1f} | {'∞':>10} | 梯度为零,学不到东西")

# 模拟训练一步后的效果
print("\n=== 训练一步后 logits 的变化(假设相同 lr) ===\n")
lr = 0.01
for label, ref_c, ref_r in [("差距大", -5.0, -15.0), ("差距小", -5.0, -5.3)]:
    # 模拟一个梯度步:policy 朝 chosen 方向移动
    logits = torch.tensor(0.0, requires_grad=True)
    loss = -F.logsigmoid(0.15 * logits)
    loss.backward()
    # 梯度大小
    grad = logits.grad.item()
    print(f"  {label}: 初始梯度 = {grad:.6f}")
    # 实际中,差距大的场景 logits 增长更快(因为 chosen/rejected 表示差异大)